In [1]:
# !pip install -q -U datasets huggingface_hub requests

import json
import os
import time
import random
import traceback
from datetime import datetime, timezone

import requests
from datasets import Dataset, load_dataset
from huggingface_hub import HfApi


In [2]:
# ## 1. CONFIG — edit everything in this cell

# ---- Hugging Face ----
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")


SYSTEM_LABEL = "finetuned"         

# INPUT_DATASET_REPO = "businessrules/Qwen_base_tuned_exp10_results"  
# OUTPUT_DATASET_REPO = "businessrules/exp10_promptC_results"  
INPUT_DATASET_REPO = "businessrules/GPT4_baseline_results"   
OUTPUT_DATASET_REPO = "businessrules/gpt4.1_promptC_results"  

# Prompt C compares the generated rules against a GOLD (human-written reference) rule

GOLD_DATASET_REPO = "businessrules/dataset_stratified_test"
GOLD_DATASET_SPLIT = "test"
GOLD_COLUMN = "br"         
ID_COLUMN = "id"                    
# OUTPUT_COLUMN = "finetuned_model_prediction"
OUTPUT_COLUMN = "gpt4_prediction"
MAX_ITEMS = None
INPUT_SPLIT = "train"

# ---- OpenRouter ----
OPENROUTER_API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")

JUDGE_MODELS = [
    "openai/gpt-4o-mini",
    "anthropic/claude-sonnet-4.6",
    "google/gemini-2.5-flash",
]

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
REQUEST_TIMEOUT = 120
TEMPERATURE = 0.0
MAX_TOKENS = 3000

# ---- Checkpointing ----
SAVE_EVERY = 10          # push to the HF output dataset after this many NEW results
MAX_RETRIES = 5          # per API call, with exponential backoff
BASE_BACKOFF_SECONDS = 5


In [3]:
# ## 1b. Verify the HF token works, and check (without writing anything) whether
# OUTPUT_DATASET_REPO already exists

_api = HfApi()
try:
    _who = _api.whoami(token=HF_TOKEN)
    print(f"HF token OK — authenticated as: {_who.get('name', _who)}")
except Exception as e:
    raise RuntimeError(
        f"HF_TOKEN check failed ({type(e).__name__}: {e}). Fix HF_TOKEN before continuing — "
        "if this fails, every push_to_hub call below will silently do nothing useful."
    )

_output_repo_exists = _api.repo_exists(OUTPUT_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
if _output_repo_exists:
    print(f"OUTPUT_DATASET_REPO already exists: https://huggingface.co/datasets/{OUTPUT_DATASET_REPO}")
else:
    print(f"OUTPUT_DATASET_REPO does not exist yet — it will be CREATED automatically "
          f"the first time save_checkpoint() successfully pushes results "
          f"(push_to_hub creates the repo if it's missing, no separate 'create' step needed).")


HF token OK — authenticated as: tarannom-s
OUTPUT_DATASET_REPO does not exist yet — it will be CREATED automatically the first time save_checkpoint() successfully pushes results (push_to_hub creates the repo if it's missing, no separate 'create' step needed).


In [4]:
# ## 2. Prompt C — system prompt and user prompt template
SYSTEM_PROMPT = """You are a strict rule-alignment auditor. You are given a GOLD (reference) list of
business rules and a GENERATED list of business rules produced by a model. Your job is
to align the two lists and judge, rule by rule, how well the generated output covers
and correctly restates the gold rules.

You are NOT checking whether rules are grounded in source code (that is done elsewhere).
Assume both lists are candidate business rules; your only job is to compare them to each
other on MEANING, not exact wording. Two rules that use different words but express the
same business logic are a match. Two rules that use similar words but change the
condition, threshold, or action are NOT a full match.

Be skeptical by default. Do not mark a match as "full" unless the condition AND the
action/consequence are both equivalent. A generated rule that gets the general topic
right but changes a number, a condition, or an actor is a partial match, not full.

You must complete the alignment step for every gold rule and every generated rule before
computing any aggregate score. Scores must be a deterministic function of your alignment
judgments (formulas provided below).

Output valid JSON only. No prose outside the JSON object. No markdown code fences."""

USER_PROMPT_TEMPLATE = """Align the generated rules against the gold (reference) rules and evaluate correctness
and coverage.

GOLD RULES (reference, treat as ground truth):
\"\"\"
{gold_rules}
\"\"\"

GENERATED OUTPUT:
\"\"\"
{generated_output}
\"\"\"

Step 1: Extract every individual rule from the generated output into a numbered list
(gen_id starting at 1). Extract every individual rule from the gold rules into a
numbered list (gold_id starting at 1), preserving the reference numbering if it is
already numbered.

Step 2: For EACH gold rule, find the single best-matching generated rule, if any exists.
Record:
- gold_id
- matched_gen_id: the gen_id of the best match, or null if no generated rule
  corresponds to this gold rule at all
- match_quality: one of "full_match", "partial_match", "no_match"
  - full_match: condition AND action/consequence both equivalent in meaning
  - partial_match: same general topic/condition but action, threshold, scope, or actor
    differs in a way that would change real-world behavior
  - no_match: no generated rule addresses this gold rule's logic at all
- discrepancy: if partial_match, briefly state exactly what differs (e.g. "gold says 48
  hours, generated says 24 hours"). If full_match or no_match, use null.

Step 3: For EACH generated rule, record:
- gen_id
- role: one of "matched" (already covered as the best match for some gold rule in Step
  2), "additional_valid" (not matched to any gold rule, but expresses a plausible
  distinct business rule not present in gold — do not judge grounding here, only
  whether it's a coherent standalone rule), "duplicate" (restates another generated
  rule's logic with no new information), or "no_gold_match" (not matched to gold and not
  clearly a valid additional rule — overlaps confusingly with something in gold without
  being a clean paraphrase, or is otherwise not classifiable as additional_valid)
- duplicate_of_gen_id: if role="duplicate", the gen_id it duplicates; else null

---

Step 4: Compute aggregates.

total_gold_count = number of gold rules
total_gen_count = number of generated rules

full_match_count = count of gold rules with match_quality="full_match"
partial_match_count = count of gold rules with match_quality="partial_match"
no_match_count = count of gold rules with match_quality="no_match"

coverage_score_raw = (full_match_count + 0.5 * partial_match_count) / total_gold_count
  # this is your recall-equivalent: how much of gold is captured, partial counted as half credit

matched_gen_count = count of generated rules with role="matched"
additional_valid_count = count of generated rules with role="additional_valid"
duplicate_count = count of generated rules with role="duplicate"
no_gold_match_count = count of generated rules with role="no_gold_match"

precision_raw = (matched_gen_count_weighted) / total_gen_count
  where matched_gen_count_weighted = full_match_count + 0.5 * partial_match_count
  (i.e. numerator is the same weighted-correct count as above, denominator is everything
  the model produced, so extra/duplicate/wrong-topic rules correctly reduce precision)

f1_raw = 0 if (coverage_score_raw + precision_raw) == 0 else
  2 * coverage_score_raw * precision_raw / (coverage_score_raw + precision_raw)

redundancy_ratio = duplicate_count / total_gen_count

Convert coverage_score_raw, precision_raw, and f1_raw each to a 1-5 score using:
  1 if value < 0.2
  2 if value < 0.4
  3 if value < 0.6
  4 if value < 0.85
  5 if value >= 0.85

redundancy_score (1-5), inverted since lower redundancy is better:
  5 if redundancy_ratio == 0
  4 if redundancy_ratio < 0.1
  3 if redundancy_ratio < 0.25
  2 if redundancy_ratio < 0.4
  1 if redundancy_ratio >= 0.4

---

Calibration reference:
- full_match requires genuine equivalence of both condition and action. Do not be lenient
  just because the topic overlaps — a changed number or changed actor is a partial_match
  at best.
- no_match on a gold rule means you found nothing in the generated output addressing it,
  even loosely. Do not force a weak match just to avoid a no_match.
- additional_valid should be used only when the rule is genuinely coherent and distinct,
  not as a default bucket to avoid the less flattering no_gold_match label.

Return ONLY a JSON object matching the schema below. Every gold rule must appear in the
alignment array and every generated rule must appear in the generated_rules array.

JSON_OUTPUT_SCHEMA:
{{
  "gold_alignment": [
    {{
      "gold_id": 1,
      "matched_gen_id": 1,
      "match_quality": "full_match",
      "discrepancy": null
    }},
    {{
      "gold_id": 2,
      "matched_gen_id": 3,
      "match_quality": "partial_match",
      "discrepancy": "gold specifies cancellation window of 48 hours; generated rule states 24 hours"
    }},
    {{
      "gold_id": 3,
      "matched_gen_id": null,
      "match_quality": "no_match",
      "discrepancy": null
    }}
  ],
  "generated_rules": [
    {{
      "gen_id": 1,
      "role": "matched",
      "duplicate_of_gen_id": null
    }},
    {{
      "gen_id": 2,
      "role": "additional_valid",
      "duplicate_of_gen_id": null
    }},
    {{
      "gen_id": 3,
      "role": "matched",
      "duplicate_of_gen_id": null
    }},
    {{
      "gen_id": 4,
      "role": "duplicate",
      "duplicate_of_gen_id": 3
    }}
  ],
  "aggregates": {{
    "total_gold_count": 3,
    "total_gen_count": 4,
    "full_match_count": 1,
    "partial_match_count": 1,
    "no_match_count": 1,
    "coverage_score_raw": 0.5,
    "coverage_score": 3,
    "matched_gen_count": 2,
    "additional_valid_count": 1,
    "duplicate_count": 1,
    "no_gold_match_count": 0,
    "precision_raw": 0.375,
    "precision_score": 2,
    "f1_raw": 0.4286,
    "f1_score": 3,
    "redundancy_ratio": 0.25,
    "redundancy_score": 3
  }},
  "overall_notes": "Free-text summary of the single largest coverage gap or discrepancy, 1-2 sentences max."
}}"""


In [ ]:
# ## 3. Load the input dataset (test split)

print(f"Loading {INPUT_DATASET_REPO} split={INPUT_SPLIT} (system={SYSTEM_LABEL}) ...")
input_ds = load_dataset(INPUT_DATASET_REPO, split=INPUT_SPLIT, token=HF_TOKEN)
print(f"Loaded {len(input_ds)} rows. Columns: {input_ds.column_names}")

if ID_COLUMN not in input_ds.column_names:
    print(f"WARNING: id_column '{ID_COLUMN}' not found — falling back to row index as id.")
if OUTPUT_COLUMN not in input_ds.column_names:
    raise ValueError(
        f"OUTPUT_COLUMN '{OUTPUT_COLUMN}' not found in {INPUT_DATASET_REPO}. "
        f"Available columns: {input_ds.column_names}"
    )
if MAX_ITEMS is not None:
    input_ds = input_ds.select(range(min(MAX_ITEMS, len(input_ds))))
    print(f"MAX_ITEMS set — using only the first {len(input_ds)} rows for this run.")

# ## 3b. Load the separate GOLD RULES dataset and build an id -> gold_rules lookup

print(f"Loading gold-rules dataset {GOLD_DATASET_REPO} split={GOLD_DATASET_SPLIT} ...")
gold_ds = load_dataset(GOLD_DATASET_REPO, split=GOLD_DATASET_SPLIT, token=HF_TOKEN)
print(f"Loaded {len(gold_ds)} rows. Columns: {gold_ds.column_names}")

if GOLD_COLUMN not in gold_ds.column_names:
    raise ValueError(
        f"GOLD_COLUMN '{GOLD_COLUMN}' not found in {GOLD_DATASET_REPO}. "
        f"Available columns: {gold_ds.column_names}"
    )

gold_has_id = ID_COLUMN in gold_ds.column_names
if not gold_has_id:
    print(f"WARNING: id_column '{ID_COLUMN}' not found in {GOLD_DATASET_REPO} — "
          f"falling back to row order for the join.")
if len(gold_ds) != len(input_ds):
    print(f"WARNING: row counts differ (input_ds={len(input_ds)}, gold_ds={len(gold_ds)}) — "
          f"positional fallback may misalign for extra/missing rows.")

gold_rules_by_id = {}
gold_rules_by_index = {}
for idx, row in enumerate(gold_ds):
    gold = row.get(GOLD_COLUMN)
    gold_rules_by_index[idx] = gold
    if gold_has_id:
        gold_rules_by_id[row.get(ID_COLUMN)] = gold


def lookup_gold_rules(item_id, idx):
    """id-based lookup first (primary join key), falls back to positional index."""
    if gold_has_id and item_id in gold_rules_by_id:
        return gold_rules_by_id[item_id], "id"
    if idx in gold_rules_by_index:
        return gold_rules_by_index[idx], "index"
    return None, "missing"

# ## 4. OpenRouter call + JSON parsing/validation helpers

def call_openrouter(model: str, system_prompt: str, user_prompt: str) -> str:
    """Calls OpenRouter chat completions, returns raw text content. Retries with
    exponential backoff on transient failures (429, 5xx, timeouts)."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],

        "response_format": {"type": "json_object"},
    }

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(
                OPENROUTER_URL, headers=headers, json=payload, timeout=REQUEST_TIMEOUT
            )
            if resp.status_code == 429 or resp.status_code >= 500:
                raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:300]}")
            resp.raise_for_status()
            data = resp.json()
            return data["choices"][0]["message"]["content"]
        except Exception as e:
            last_err = e
            sleep_s = BASE_BACKOFF_SECONDS * (2 ** (attempt - 1)) + random.uniform(0, 2)
            print(f"  [retry {attempt}/{MAX_RETRIES}] {model} call failed: {e}. "
                  f"Sleeping {sleep_s:.1f}s ...")
            time.sleep(sleep_s)
    raise RuntimeError(f"OpenRouter call failed after {MAX_RETRIES} retries: {last_err}")


def strip_code_fences(text: str) -> str:
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[1] if "\n" in t else t
        if t.endswith("```"):
            t = t.rsplit("```", 1)[0]
    return t.strip()


def _ratio_score(ratio: float) -> int:
    if ratio < 0.2:
        return 1
    if ratio < 0.4:
        return 2
    if ratio < 0.6:
        return 3
    if ratio < 0.85:
        return 4
    return 5


def _redundancy_score(ratio: float) -> int:
    if ratio == 0:
        return 5
    if ratio < 0.1:
        return 4
    if ratio < 0.25:
        return 3
    if ratio < 0.4:
        return 2
    return 1


def recompute_checks(parsed: dict) -> dict:
    """Recomputes all Prompt C aggregates (coverage/precision/F1/redundancy, raw +
    1-5 scores) from the per-rule gold_alignment / generated_rules judgments, per the
    prompt's own formulas. Flags mismatches vs what the model reported in its own
    'aggregates' block — this is the reliability check, same idea as Prompt A/B."""
    issues = []

    gold_alignment = parsed.get("gold_alignment", [])
    generated_rules = parsed.get("generated_rules", [])

    total_gold_count = len(gold_alignment)
    total_gen_count = len(generated_rules)

    full_match_count = sum(1 for r in gold_alignment if r.get("match_quality") == "full_match")
    partial_match_count = sum(1 for r in gold_alignment if r.get("match_quality") == "partial_match")
    no_match_count = sum(1 for r in gold_alignment if r.get("match_quality") == "no_match")

    matched_gen_count = sum(1 for r in generated_rules if r.get("role") == "matched")
    additional_valid_count = sum(1 for r in generated_rules if r.get("role") == "additional_valid")
    duplicate_count = sum(1 for r in generated_rules if r.get("role") == "duplicate")
    no_gold_match_count = sum(1 for r in generated_rules if r.get("role") == "no_gold_match")

    coverage_score_raw = (
        (full_match_count + 0.5 * partial_match_count) / total_gold_count
        if total_gold_count > 0 else 0.0
    )

    matched_gen_count_weighted = full_match_count + 0.5 * partial_match_count
    precision_raw = (
        matched_gen_count_weighted / total_gen_count
        if total_gen_count > 0 else 0.0
    )

    f1_raw = (
        0.0 if (coverage_score_raw + precision_raw) == 0
        else 2 * coverage_score_raw * precision_raw / (coverage_score_raw + precision_raw)
    )

    redundancy_ratio = (duplicate_count / total_gen_count) if total_gen_count > 0 else 0.0

    recomputed = {
        "total_gold_count": total_gold_count,
        "total_gen_count": total_gen_count,
        "full_match_count": full_match_count,
        "partial_match_count": partial_match_count,
        "no_match_count": no_match_count,
        "coverage_score_raw": coverage_score_raw,
        "coverage_score": _ratio_score(coverage_score_raw),
        "matched_gen_count": matched_gen_count,
        "additional_valid_count": additional_valid_count,
        "duplicate_count": duplicate_count,
        "no_gold_match_count": no_gold_match_count,
        "precision_raw": precision_raw,
        "precision_score": _ratio_score(precision_raw),
        "f1_raw": f1_raw,
        "f1_score": _ratio_score(f1_raw),
        "redundancy_ratio": redundancy_ratio,
        "redundancy_score": _redundancy_score(redundancy_ratio),
    }

    agg = parsed.get("aggregates", {})
    if not isinstance(agg, dict):
        agg = {}
        issues.append("aggregates field malformed")

    for key, recomputed_val in recomputed.items():
        model_val = agg.get(key)
        if isinstance(recomputed_val, float):
            mismatch = (
                model_val is None
                or not isinstance(model_val, (int, float))
                or abs(float(model_val) - recomputed_val) > 1e-6
            )
        else:
            mismatch = model_val != recomputed_val
        if mismatch:
            issues.append(
                f"{key} mismatch: model said {model_val}, recomputed {recomputed_val}"
            )

 
    if agg.get("coverage_score") == 5 and (no_match_count > 0 or partial_match_count > 0):
        issues.append(
            "CONTRADICTION: coverage_score=5 but at least one gold rule is not a full_match"
        )

    return {
        "recomputed_total_gold_count": recomputed["total_gold_count"],
        "recomputed_total_gen_count": recomputed["total_gen_count"],
        "recomputed_full_match_count": recomputed["full_match_count"],
        "recomputed_partial_match_count": recomputed["partial_match_count"],
        "recomputed_no_match_count": recomputed["no_match_count"],
        "recomputed_coverage_score_raw": recomputed["coverage_score_raw"],
        "recomputed_coverage_score": recomputed["coverage_score"],
        "recomputed_matched_gen_count": recomputed["matched_gen_count"],
        "recomputed_additional_valid_count": recomputed["additional_valid_count"],
        "recomputed_duplicate_count": recomputed["duplicate_count"],
        "recomputed_no_gold_match_count": recomputed["no_gold_match_count"],
        "recomputed_precision_raw": recomputed["precision_raw"],
        "recomputed_precision_score": recomputed["precision_score"],
        "recomputed_f1_raw": recomputed["f1_raw"],
        "recomputed_f1_score": recomputed["f1_score"],
        "recomputed_redundancy_ratio": recomputed["redundancy_ratio"],
        "recomputed_redundancy_score": recomputed["redundancy_score"],
        "validation_issues": issues,
        "is_consistent": len(issues) == 0,
    }


def judge_one(model: str, gold_rules: str, generated_output: str) -> dict:
    """Runs Prompt C once and returns a flat result dict (never raises — errors are
    captured in the row so the pipeline keeps going)."""
    user_prompt = USER_PROMPT_TEMPLATE.format(
        gold_rules=gold_rules, generated_output=generated_output
    )
    raw = None
    try:
        raw = call_openrouter(model, SYSTEM_PROMPT, user_prompt)
        cleaned = strip_code_fences(raw)
        parsed = json.loads(cleaned)
        checks = recompute_checks(parsed)
        return {
            "status": "ok",
            "raw_response": raw,
            "parsed_json": json.dumps(parsed, ensure_ascii=False),
            "error": None,
            **checks,
        }
    except Exception as e:
        return {
            "status": "error",
            "raw_response": raw,
            "parsed_json": None,
            "error": f"{type(e).__name__}: {e}",
            "recomputed_total_gold_count": None,
            "recomputed_total_gen_count": None,
            "recomputed_full_match_count": None,
            "recomputed_partial_match_count": None,
            "recomputed_no_match_count": None,
            "recomputed_coverage_score_raw": None,
            "recomputed_coverage_score": None,
            "recomputed_matched_gen_count": None,
            "recomputed_additional_valid_count": None,
            "recomputed_duplicate_count": None,
            "recomputed_no_gold_match_count": None,
            "recomputed_precision_raw": None,
            "recomputed_precision_score": None,
            "recomputed_f1_raw": None,
            "recomputed_f1_score": None,
            "recomputed_redundancy_ratio": None,
            "recomputed_redundancy_score": None,
            "validation_issues": [str(e)],
            "is_consistent": False,
        }


In [6]:
# ## 5. Resume support — load existing output dataset (if any) and skip done work

from huggingface_hub import HfApi, create_repo
from datasets import load_dataset, Dataset
import json
import time
import random
import requests
import traceback
from datetime import datetime, timezone

def ensure_repo_exists(repo_id, token):
    """Ensures the dataset repository exists on HF Hub before attempting operations."""
    api = HfApi(token=token)
    try:
        api.repo_info(repo_id=repo_id, repo_type="dataset")
    except Exception:
        print(f"Dataset {repo_id} not found on Hub. Creating repository...")
        create_repo(repo_id=repo_id, repo_type="dataset", token=token, private=True)
        print(f"Repository {repo_id} created successfully.")

def load_existing_results():
    ensure_repo_exists(OUTPUT_DATASET_REPO, token=HF_TOKEN)
    try:
        existing = load_dataset(OUTPUT_DATASET_REPO, split="train", token=HF_TOKEN)
        results = existing.to_list()
        done_keys = {(r["item_id"], r["system"], r["judge_model"]) for r in results}
        print(f"Resuming: found {len(results)} existing results "
              f"({len(done_keys)} unique combos) in {OUTPUT_DATASET_REPO}.")
        return results, done_keys
    except Exception as e:
        print(f"Starting fresh (dataset is empty or uninitialized): {e}")
        return [], set()

all_results, done_keys = load_existing_results()


Dataset businessrules/gpt4.1_promptC_results not found on Hub. Creating repository...
Repository businessrules/gpt4.1_promptC_results created successfully.
Starting fresh (dataset is empty or uninitialized): The directory at hf://datasets/businessrules/gpt4.1_promptC_results@3e6aee61be8a0802b410b8b6a3fdc8aacd70f943 doesn't contain any data files


In [7]:
# ## 6. Build the full work queue: item x judge_model (for SYSTEM_LABEL)

work_items = []
for idx, row in enumerate(input_ds):
    item_id = row.get(ID_COLUMN) if (ID_COLUMN in input_ds.column_names and row.get(ID_COLUMN) is not None) else idx
    generated_output = row.get(OUTPUT_COLUMN)

    # Retrieve gold rules using lookup function from Section 3b
    gold_rules, gold_type = lookup_gold_rules(item_id, idx)

    if generated_output is None or gold_rules is None:
        print(f"Skipping row {idx} (item_id={item_id}): output missing={generated_output is None}, gold missing={gold_rules is None}")
        continue

    for model in JUDGE_MODELS:
        key = (item_id, SYSTEM_LABEL, model)
        if key in done_keys:
            continue
        work_items.append({
            "item_id": item_id,
            "system": SYSTEM_LABEL,
            "judge_model": model,
            "generated_output": generated_output,
            "gold_rules": gold_rules,
        })

print(f"Total work items remaining: {len(work_items)}")


Total work items remaining: 600


In [8]:
# ## 7. Checkpoint save helper

def save_checkpoint(results):
    if not results:
        return
    ds = Dataset.from_list(results)
    ds.push_to_hub(OUTPUT_DATASET_REPO, token=HF_TOKEN)
    print(f"   [checkpoint] pushed {len(results)} total results to {OUTPUT_DATASET_REPO}")


In [ ]:
# ## 8. Main loop — judges every work item, checkpointing every SAVE_EVERY new results

new_since_last_save = 0

try:
    for i, item in enumerate(work_items, start=1):
        print(f"[{i}/{len(work_items)}] item_id={item['item_id']} "
              f"system={item['system']} model={item['judge_model']}")

        result = judge_one(item["judge_model"], item["gold_rules"], item["generated_output"])
        row = {
            "item_id": item["item_id"],
            "system": item["system"],
            "judge_model": item["judge_model"],
            "timestamp": datetime.now(timezone.utc).isoformat(),
            **result,
            # keep validation_issues as a JSON string for a clean Arrow schema
            "validation_issues": json.dumps(result["validation_issues"]),
        }
        all_results.append(row)
        new_since_last_save += 1

        if new_since_last_save >= SAVE_EVERY:
            save_checkpoint(all_results)
            new_since_last_save = 0

except KeyboardInterrupt:
    print("Interrupted — saving progress before exiting.")
except Exception:
    print("Unexpected error — saving progress before re-raising.")
    print(traceback.format_exc())
finally:
    if new_since_last_save > 0:
        save_checkpoint(all_results)

print(f"Done. {len(all_results)} total results saved to {OUTPUT_DATASET_REPO}.")


In [ ]:
# ## 9. Quick sanity check on judge reliability

inconsistent = [r for r in all_results if not r.get("is_consistent", True)]
print(f"{len(inconsistent)}/{len(all_results)} results had aggregate-vs-recomputed mismatches.")
if inconsistent:
    print("Example issues from the first few:")
    for r in inconsistent[:5]:
        print(f" - item_id={r['item_id']} system={r['system']} model={r['judge_model']}: "
              f"{r['validation_issues']}")
